In [ ]:
!pip install -q google-genai pydantic

In [ ]:
# Task 4.1 — One validated call (enter API key at runtime; clear outputs before uploading to GitHub)
import json
import time
import pandas as pd
from google import genai
from google.genai import types
from google.genai.errors import APIError

API_KEY = input("Enter the API Key: ")
print("API key received; initializing client...")
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(timeout=60_000),  # 60 seconds in ms
)


MODEL_NAME = "gemini-3.1-flash-lite"  # Replace with your Session 22.1 or active free Flash model in AI Studio if needed

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",

    "illness_or_emergency",
    "work_or_schedule",
    "other",
]

SYSTEM_TEXT = "You are a clinical operations analyst classifying patient no-show call-centre notes for CareLoop Clinics."

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"},
    },
    "required": ["root_cause", "confidence", "summary", "recommended_action"],
}

api_call_count = 0

def build_craft_prompt(note_text: str) -> str:
    return f"""Context: CareLoop Clinics operates eight outpatient clinics and collects short call-centre follow-up notes after missed appointments.
Role: You are an analyst extracting structured reasons from a single call-centre note.
Action: Choose root_cause strictly from {ALLOWED_ROOT_CAUSES} (choose "other" if unclear or ambiguous rather than guessing), estimate confidence in [0, 1], write a summary of at most 25 words, and give a recommended_action.
Tone: Concise, objective, and factual.
Format: Return JSON matching the required schema.

Note:
{note_text}
"""

def call_gemini_raw(prompt_text: str) -> str:
    global api_call_count
    api_call_count += 1
    base_cfg = dict(
        system_instruction=SYSTEM_TEXT,
        temperature=0,
        max_output_tokens=1000,
        response_mime_type="application/json",
        response_schema=SCHEMA,
    )
    try:
        resp = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(
                **base_cfg,
                thinking_config=types.ThinkingConfig(thinking_level="minimal"),
            ),
        )
    except (APIError, TypeError, ValueError):
        resp = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt_text,
            config=types.GenerateContentConfig(**base_cfg),
        )
    return resp.text

def analyse_note(note_text: str) -> dict:
    return json.loads(call_gemini_raw(build_craft_prompt(note_text)))

notes_df = pd.read_csv("followup_notes.csv")
n001_text = notes_df.loc[notes_df["note_id"] == "N001", "note_text"].iloc[0]
n001_out = analyse_note(n001_text)
print(n001_out)

In [ ]:
# Task 4.2 — Validate and process first 10 notes -> notes_analysed.csv
from pydantic import BaseModel, Field, ValidationError

class FollowupInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

first_10 = notes_df.head(10).copy()
rows_out = []
calls_4_2 = 0

for _, r in first_10.iterrows():
    nid, aid, ntxt = r["note_id"], r["appointment_id"], r["note_text"]
    if pd.isna(ntxt) or str(ntxt).strip() == "":
        rows_out.append({
            "note_id": nid, "appointment_id": aid,
            "root_cause": "other", "confidence": 0.0,
            "summary": "No note recorded", "recommended_action": "Re-call patient",
            "validation_status": "skipped_blank", "raw_response": ""
        })
        continue
    if calls_4_2 > 0:
        time.sleep(13)
    raw_resp = ""
    try:
        raw_resp = call_gemini_raw(build_craft_prompt(str(ntxt)))
        calls_4_2 += 1
        parsed = json.loads(raw_resp)
        obj = FollowupInsight(**parsed)
        is_ok = obj.root_cause in ALLOWED_ROOT_CAUSES
        rows_out.append({
            "note_id": nid, "appointment_id": aid,
            "root_cause": parsed.get("root_cause", ""),
            "confidence": parsed.get("confidence", ""),
            "summary": parsed.get("summary", ""),
            "recommended_action": parsed.get("recommended_action", ""),
            "validation_status": "valid" if is_ok else "invalid",
            "raw_response": raw_resp
        })
    except Exception:
        parsed = {}
        try:
            parsed = json.loads(raw_resp)
        except Exception:
            pass
        rows_out.append({
            "note_id": nid, "appointment_id": aid,
            "root_cause": parsed.get("root_cause", "") if isinstance(parsed, dict) else "",
            "confidence": parsed.get("confidence", "") if isinstance(parsed, dict) else "",
            "summary": parsed.get("summary", "") if isinstance(parsed, dict) else "",
            "recommended_action": parsed.get("recommended_action", "") if isinstance(parsed, dict) else "",
            "validation_status": "invalid",
            "raw_response": raw_resp
        })

notes_analysed = pd.DataFrame(rows_out)
notes_analysed.to_csv("notes_analysed.csv", index=False)
print(notes_analysed[["note_id", "root_cause", "confidence", "validation_status"]])

In [ ]:
# Task 4.3 & 4.4 — Deliberate ValidationErrors, V1 vs V2 comparison, call count & cross-tab
for bad in [
    {"root_cause": "other", "confidence": 1.7, "summary": "Over confidence", "recommended_action": "Follow up"},
    {"root_cause": "other", "confidence": 0.8, "summary": "Missing action"}
]:
    try:
        FollowupInsight(**bad)
    except ValidationError as err:
        print("ValidationError:", err, "\n")

comp = []
for nid in ["N007", "N009"]:
    ntxt = notes_df.loc[notes_df["note_id"] == nid, "note_text"].iloc[0]
    time.sleep(13)
    v1_parsed = json.loads(call_gemini_raw(f"Classify this note: {ntxt}"))
    v1_rc = v1_parsed.get("root_cause", "")
    v2_rc = notes_analysed.loc[notes_analysed["note_id"] == nid, "root_cause"].iloc[0]
    comp.append({
        "note_id": nid,
        "V1_root_cause": v1_rc, "V1_in_allowed": v1_rc in ALLOWED_ROOT_CAUSES,
        "V2_root_cause": v2_rc, "V2_in_allowed": v2_rc in ALLOWED_ROOT_CAUSES,
    })
print(pd.DataFrame(comp).to_string(index=False))
print("Deploy V2 because it enforces allowed categories and avoids guessing on ambiguous notes.")
print("Total API calls used across Tasks 4.1–4.3:", api_call_count)

appts_clean = pd.read_csv("appointments_clean.csv")
merged = notes_analysed.merge(appts_clean, on="appointment_id", how="left")
print("\nCross-tab root_cause vs reminder_sent:\n", pd.crosstab(merged["root_cause"], merged["reminder_sent"], margins=True))